# Milestone 2 – Eksploracja danych i analiza cech (MAPB)

Notebook realizuje wymagania Milestone 2:
- przygotowanie i normalizacja logu zdarzeń,
- wykrywanie wartości odstąjących (outlierów),
- redukcja wymiarowości: PCA, t-SNE,
- klasteryzacja eventów i aktywności,
- analiza relacji między zdarzeniami (macierz korelacji, DFG),
- wzorce czasowe (odstępy między eventami),
- analiza sekwencji i wzorce sygnałowe (CEP patterns),
- wykrywanie anomalii (Isolation Forest, LOF).


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
np.random.seed(SEED)
print('Biblioteki zaladowane.')


## 1. Wczytanie i przygotowanie danych

Wczytujemy pliki `Signature_*.txt` identycznie jak w Milestone 1,
następnie wykonujemy normalizację i inżynierię cech.


In [ ]:
candidate_dirs = [Path('.'), Path('Dataset_8087219'), Path('..') / 'Dataset_8087219']
DATA_DIR = None
for d in candidate_dirs:
    if (d / 'Signature_Burn.txt').exists():
        DATA_DIR = d.resolve()
        break
if DATA_DIR is None:
    raise FileNotFoundError('Nie znaleziono folderu z plikami Signature_*.txt.')

signature_files = sorted(DATA_DIR.glob('Signature_*.txt'))
print(f'DATA_DIR: {DATA_DIR}')
print(f'Pliki: {[f.name for f in signature_files]}')


In [ ]:
records = []
for file_path in signature_files:
    activity = file_path.stem.replace('Signature_', '')
    case_id  = f'case_{activity}_001'
    with file_path.open('r', encoding='utf-8') as f:
        for idx, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            row = {'case_id': case_id, 'activity': activity,
                   'event_index': idx, 'station': obj.get('station'),
                   'timestamp_raw': obj.get('timestamp')}
            for k, v in obj.get('events', {}).items():
                row[k] = v
            records.append(row)

df = pd.DataFrame(records)
df['timestamp'] = pd.to_datetime(df['timestamp_raw'], utc=True, errors='coerce')
META_COLS = ['case_id','activity','event_index','station','timestamp_raw','timestamp']
SIGNAL_COLS = [c for c in df.columns if c not in META_COLS]
activities = sorted(df['activity'].unique())

print(f'Wczytano {len(df)} eventow, {len(SIGNAL_COLS)} kolumn sygnalow.')
print(f'Aktywnosci: {activities}')
df.head(3)


## 2. Czyszczenie i normalizacja danych

**Strategia NaN:** każda stacja raportuje tylko swoje sygnały – NaN jest strukturalne (nie błąd). Do analiz wielowymiarowych wypełniamy NaN wartością `0` (sygnał nieaktywny = 0), co jest semantycznie poprawne.


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

df_filled = df.copy()
df_filled[SIGNAL_COLS] = df_filled[SIGNAL_COLS].fillna(0)

X_raw    = df_filled[SIGNAL_COLS].values.astype(float)
X_minmax = MinMaxScaler().fit_transform(X_raw)
X_std    = StandardScaler().fit_transform(X_raw)

print('Ksztalt macierzy cech:', X_raw.shape)
print('Wartosci unikalne w surowych sygnalach:', sorted(np.unique(X_raw)))

# Tabela: liczba NaN per kolumna sygnalu
nan_counts = df[SIGNAL_COLS].isna().sum()
nan_df = pd.DataFrame({'kolumna': nan_counts.index,
                       'NaN': nan_counts.values,
                       'aktywne_dla': [', '.join(df[df[c].notna()]['activity'].unique())
                                       for c in SIGNAL_COLS]})
print('\nPokrycie sygnalow (NaN = sygnal nieaktywny dla danej stacji):')
print(nan_df.to_string(index=False))


## 3. Wykrywanie wartości odstąjących (outlierów)

### 3.1 Odstępy czasowe między eventami

Sprawdzamy regularność próbkowania (~2 s). Outlier = odstęp > 3 s lub < 1 s.


In [ ]:
df_s = df.sort_values(['activity','event_index']).copy()
df_s['dt_s'] = df_s.groupby('activity')['timestamp'].diff().dt.total_seconds()

print('=== STATYSTYKI ODSTEPOW CZASOWYCH PER AKTYWNOSC ===')
stats = df_s.groupby('activity')['dt_s'].agg(
    n='count', mean='mean', std='std', min='min', max='max', median='median'
).round(3)
print(stats.to_string())

outliers_t = df_s[df_s['dt_s'].notna() & ((df_s['dt_s'] > 3) | (df_s['dt_s'] < 1))]
n_total_gaps = len(df) - df['activity'].nunique()
print(f'\nOutliery czasowe: {len(outliers_t)} z {n_total_gaps} odstepow '
      f'({len(outliers_t)/n_total_gaps*100:.1f}%)')
if len(outliers_t) > 0:
    print(outliers_t[['activity','event_index','timestamp_raw','dt_s']].to_string(index=False))


In [ ]:
colors_act = plt.cm.Set1.colors
color_map  = {a: colors_act[i] for i, a in enumerate(activities)}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, act in enumerate(activities):
    sub = df_s[df_s['activity'] == act]['dt_s'].dropna()
    ax  = axes[i]
    ax.bar(range(len(sub)), sub.values, color=color_map[act], edgecolor='white', alpha=0.85)
    ax.axhline(2, color='green', linestyle='--', lw=1.5, label='oczekiwane 2s')
    ax.axhline(sub.mean(), color='red', linestyle=':', lw=1.5,
               label=f'srednia={sub.mean():.2f}s')
    om = (sub > 3) | (sub < 1)
    if om.any():
        ax.scatter(np.where(om)[0], sub.values[om], color='red', s=80, zorder=5, label='outlier')
    ax.set_title(f'{act}\n(n={len(sub)} odstepow)', fontsize=10, fontweight='bold')
    ax.set_xlabel('Nr odstępu'); ax.set_ylabel('Czas [s]')
    ax.set_ylim(0, max(sub.max() * 1.2, 4))
    ax.legend(fontsize=7)
plt.suptitle('Odstepy czasowe miedzy eventami per aktywnosc', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('m2_time_gaps.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_time_gaps.png')


### 3.2 Outliery w wartościach sygnałów (Isolation Forest)

Isolation Forest wykrywa eventy o nietypowym profilu sygnałowym.


In [ ]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.05, random_state=SEED)
iso_labels = iso.fit_predict(X_std)
iso_scores = iso.score_samples(X_std)
df_filled['iso_label'] = iso_labels
df_filled['iso_score'] = iso_scores

n_anom = (iso_labels == -1).sum()
print(f'Isolation Forest: {n_anom} anomalii (5% contamination)')
print('\nRozklad anomalii per aktywnosc:')
print(df_filled.groupby('activity')['iso_label']
      .apply(lambda x: (x == -1).sum()).rename('n_anomalii').to_string())
print('\nSzczegoly anomalii:')
anom = df_filled[df_filled['iso_label'] == -1][['activity','event_index','station','timestamp_raw','iso_score']]
print(anom.to_string(index=False))

fig, ax = plt.subplots(figsize=(13, 4))
bar_colors = ['red' if l == -1 else 'steelblue' for l in iso_labels]
ax.bar(range(len(iso_scores)), iso_scores, color=bar_colors, edgecolor='none')
threshold = iso_scores[iso_labels == -1].max() if n_anom > 0 else -0.1
ax.axhline(threshold, color='red', linestyle='--', lw=1.5, label=f'prog anomalii ({threshold:.3f})')
# Etykiety aktywnosci
pos = 0
for act in activities:
    n = (df['activity'] == act).sum()
    ax.axvline(pos, color='gray', alpha=0.4, lw=1)
    ax.text(pos + n/2, ax.get_ylim()[0] * 0.98, act, ha='center', fontsize=8,
            rotation=30, va='top', color='black')
    pos += n
ax.set_xlabel('Event (indeks globalny)')
ax.set_ylabel('Isolation Forest score')
ax.set_title('Isolation Forest – score anomalii per event (czerwony = anomalia)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('m2_isolation_forest.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_isolation_forest.png')


## 4. Redukcja wymiarowości

### 4.1 PCA (Principal Component Analysis)

PCA redukuje 26-wymiarową przestrzeń sygnałów do 2D/3D. Sprawdzamy, czy eventy różnych aktywności tworzą odrębne skupiska.


In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA(random_state=SEED)
pca_full.fit(X_minmax)
explained  = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

print('=== WARIANCJA WYJASNIANIA PRZEZ KOLEJNE KOMPONENTY PCA ===')
for i, (ev, cum) in enumerate(zip(explained[:10], cumulative[:10])):
    print(f'  PC{i+1:2d}: {ev*100:5.1f}%  (kumulatywnie: {cum*100:5.1f}%)')

n_80 = int(np.argmax(cumulative >= 0.80)) + 1
n_95 = int(np.argmax(cumulative >= 0.95)) + 1
print(f'\nKomponenty do 80% wariancji: {n_80}')
print(f'Komponenty do 95% wariancji: {n_95}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
n_show = min(14, len(explained))
axes[0].bar(range(1, n_show+1), explained[:n_show]*100, color='steelblue', edgecolor='white')
for i, v in enumerate(explained[:n_show]*100):
    axes[0].text(i+1, v+0.3, f'{v:.1f}%', ha='center', va='bottom', fontsize=8)
axes[0].set_xlabel('Komponent PCA'); axes[0].set_ylabel('Wariancja [%]')
axes[0].set_title('Scree plot', fontweight='bold')

axes[1].plot(range(1, len(cumulative)+1), cumulative*100, 'o-', color='steelblue', lw=2)
axes[1].axhline(80, color='orange', linestyle='--', label='80%')
axes[1].axhline(95, color='red',    linestyle='--', label='95%')
axes[1].axvline(n_80, color='orange', linestyle=':', alpha=0.7)
axes[1].axvline(n_95, color='red',    linestyle=':', alpha=0.7)
axes[1].set_xlabel('Liczba komponentow'); axes[1].set_ylabel('Kumulatywna wariancja [%]')
axes[1].set_title('Kumulatywna wariancja', fontweight='bold')
axes[1].legend(); axes[1].set_ylim(0, 105)
plt.tight_layout()
plt.savefig('m2_pca_scree.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_pca_scree.png')


In [ ]:
pca2 = PCA(n_components=2, random_state=SEED)
X_pca2 = pca2.fit_transform(X_minmax)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter PC1 vs PC2
for act in activities:
    mask = df['activity'] == act
    axes[0].scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                    c=[color_map[act]], label=act, s=60, alpha=0.85,
                    edgecolors='white', linewidths=0.5)
axes[0].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}% wariancji)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}% wariancji)', fontsize=11)
axes[0].set_title('PCA 2D – eventy per aktywnosc', fontweight='bold')
axes[0].legend(fontsize=9)

# Biplot
loadings = pca2.components_.T
top8 = np.argsort(loadings[:,0]**2 + loadings[:,1]**2)[-8:]
for act in activities:
    mask = df['activity'] == act
    axes[1].scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                    c=[color_map[act]], label=act, s=40, alpha=0.5, edgecolors='none')
for i in top8:
    axes[1].arrow(0, 0, loadings[i,0]*2, loadings[i,1]*2,
                  head_width=0.05, head_length=0.03, fc='black', ec='black', lw=1.5)
    axes[1].text(loadings[i,0]*2.15, loadings[i,1]*2.15, SIGNAL_COLS[i],
                 fontsize=8, ha='center', color='darkred', fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
axes[1].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
axes[1].set_title('PCA Biplot – 8 najwazniejszych sygnalow', fontweight='bold')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig('m2_pca_2d.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_pca_2d.png')

print('\n=== LADUNKI PC1 I PC2 (top 10) ===')
ld = pd.DataFrame(loadings, index=SIGNAL_COLS, columns=['PC1','PC2'])
ld['magnitude'] = np.sqrt(ld['PC1']**2 + ld['PC2']**2)
print(ld.sort_values('magnitude', ascending=False).head(10).round(3).to_string())


### 4.2 t-SNE

t-SNE lepiej zachowuje lokalne struktury skupisk niż PCA.
Perplexity=15 dobrana do małego zbioru (n=119).


In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=15, n_iter=2000,
            random_state=SEED, learning_rate='auto', init='pca')
X_tsne = tsne.fit_transform(X_minmax)

fig, ax = plt.subplots(figsize=(10, 7))
for act in activities:
    mask = df['activity'] == act
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               c=[color_map[act]], label=act, s=70, alpha=0.85,
               edgecolors='white', linewidths=0.5)
    cx, cy = X_tsne[mask, 0].mean(), X_tsne[mask, 1].mean()
    ax.text(cx, cy, act, fontsize=9, fontweight='bold', ha='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color_map[act], alpha=0.3))
ax.set_xlabel('t-SNE dim 1', fontsize=11)
ax.set_ylabel('t-SNE dim 2', fontsize=11)
ax.set_title('t-SNE 2D – eventy per aktywnosc (perplexity=15)', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('m2_tsne.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_tsne.png')


## 5. Klasteryzacja

### 5.1 K-Means na przestrzeni PCA (5 komponentów)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

pca5 = PCA(n_components=5, random_state=SEED)
X_pca5 = pca5.fit_transform(X_minmax)

inertias, silhouettes = [], []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    lbl = km.fit_predict(X_pca5)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca5, lbl))

best_k = list(K_range)[int(np.argmax(silhouettes))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'o-', color='steelblue', lw=2)
for k, v in zip(K_range, inertias):
    axes[0].text(k, v + max(inertias)*0.01, f'{v:.0f}', ha='center', fontsize=8)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inercja (WCSS)')
axes[0].set_title('Metoda lokcia (Elbow)', fontweight='bold')

axes[1].plot(K_range, silhouettes, 'o-', color='coral', lw=2)
axes[1].axvline(best_k, color='red', linestyle='--', label=f'najlepsze k={best_k}')
for k, v in zip(K_range, silhouettes):
    axes[1].text(k, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette score vs k', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('m2_kmeans_selection.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Najlepsze k wg silhouette: {best_k} (score={max(silhouettes):.3f})')


In [ ]:
# K-Means k=6 i k=best_k
for k_use in [6, best_k]:
    km = KMeans(n_clusters=k_use, random_state=SEED, n_init=20)
    cl  = km.fit_predict(X_pca5)
    ari = adjusted_rand_score(df['activity'], cl)
    sil = silhouette_score(X_pca5, cl)
    print(f'\n=== K-Means k={k_use} ===')
    print(f'  ARI (vs aktywnosci): {ari:.3f}')
    print(f'  Silhouette:          {sil:.3f}')
    ct = pd.crosstab(df['activity'], cl,
                     rownames=['Aktywnosc'], colnames=['Klaster'])
    print(ct.to_string())

# Wizualizacja k=6 vs rzeczywiste aktywnosci
km6 = KMeans(n_clusters=6, random_state=SEED, n_init=20)
cl6 = km6.fit_predict(X_pca5)
cluster_colors = plt.cm.tab10.colors

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for c in range(6):
    mask = cl6 == c
    axes[0].scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                    c=[cluster_colors[c]], label=f'Klaster {c}',
                    s=60, alpha=0.85, edgecolors='white')
axes[0].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('K-Means k=6 na PCA 2D', fontweight='bold')
axes[0].legend(fontsize=9)

for act in activities:
    mask = df['activity'] == act
    axes[1].scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                    c=[color_map[act]], label=act, s=60, alpha=0.85, edgecolors='white')
axes[1].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('Rzeczywiste aktywnosci na PCA 2D', fontweight='bold')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig('m2_kmeans_vs_activities.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_kmeans_vs_activities.png')


### 5.2 Klasteryzacja hierarchiczna i dendrogram aktywności


In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform

# Profile aktywnosci: srednia wartosci sygnalow
act_profiles = df_filled.groupby('activity')[SIGNAL_COLS].mean()
print('=== PROFILE SYGNALOWE AKTYWNOSCI (srednie) ===')
print(act_profiles.round(2).to_string())

Z = linkage(act_profiles.values, method='ward')
fig, ax = plt.subplots(figsize=(10, 5))
dendrogram(Z, labels=act_profiles.index.tolist(), ax=ax,
           color_threshold=0.7*max(Z[:,2]), leaf_font_size=12)
ax.set_title('Dendrogram hierarchicznej klasteryzacji aktywnosci\n'
             '(Ward linkage, przestrzen sygnalow)', fontweight='bold')
ax.set_xlabel('Aktywnosc'); ax.set_ylabel('Odleglosc (Ward)')
plt.tight_layout()
plt.savefig('m2_dendrogram.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_dendrogram.png')

dist_m = pd.DataFrame(squareform(pdist(act_profiles.values, metric='euclidean')),
                       index=act_profiles.index, columns=act_profiles.index)
print('\n=== MACIERZ ODLEGLOSCI EUKLIDESOWYCH MIEDZY AKTYWNOSCIAMI ===')
print(dist_m.round(1).to_string())


## 6. Analiza relacji między zdarzeniami

### 6.1 Macierz korelacji Spearmana (globalnie)


In [ ]:
corr = df_filled[SIGNAL_COLS].corr(method='spearman')

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label='Korelacja Spearmana')
ax.set_xticks(range(len(SIGNAL_COLS)))
ax.set_yticks(range(len(SIGNAL_COLS)))
ax.set_xticklabels(SIGNAL_COLS, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(SIGNAL_COLS, fontsize=8)
ax.set_title('Macierz korelacji Spearmana – sygnalow (NaN->0)', fontweight='bold', fontsize=12)
for i in range(len(SIGNAL_COLS)):
    for j in range(len(SIGNAL_COLS)):
        v = corr.values[i, j]
        if abs(v) > 0.5 and i != j:
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6,
                    color='white' if abs(v) > 0.7 else 'black', fontweight='bold')
plt.tight_layout()
plt.savefig('m2_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_correlation_heatmap.png')

pairs = []
for i in range(len(SIGNAL_COLS)):
    for j in range(i+1, len(SIGNAL_COLS)):
        v = corr.values[i, j]
        if abs(v) > 0.5:
            pairs.append({'s1': SIGNAL_COLS[i], 's2': SIGNAL_COLS[j], 'r': round(v, 3)})
pairs_df = pd.DataFrame(pairs).sort_values('r', key=abs, ascending=False)
print('\n=== SILNE KORELACJE (|r| > 0.5) ===')
print(pairs_df.to_string(index=False) if len(pairs_df) > 0 else 'Brak.')


### 6.2 Korelacje wewnątrz aktywności


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
for idx, act in enumerate(activities):
    sub = df[df['activity'] == act]
    asigs = [c for c in SIGNAL_COLS if sub[c].notna().any()]
    corr_a = sub[asigs].fillna(0).corr(method='spearman')
    im = axes[idx].imshow(corr_a.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    axes[idx].set_xticks(range(len(asigs)))
    axes[idx].set_yticks(range(len(asigs)))
    axes[idx].set_xticklabels(asigs, rotation=45, ha='right', fontsize=7)
    axes[idx].set_yticklabels(asigs, fontsize=7)
    axes[idx].set_title(f'{act} ({sub["station"].iloc[0]})\n{len(asigs)} sygnalow',
                        fontweight='bold', fontsize=10)
    for i in range(len(asigs)):
        for j in range(len(asigs)):
            v = corr_a.values[i, j]
            if abs(v) > 0.4 and i != j:
                axes[idx].text(j, i, f'{v:.2f}', ha='center', va='center',
                               fontsize=6, color='white' if abs(v) > 0.7 else 'black')
    plt.colorbar(im, ax=axes[idx], shrink=0.8)
plt.suptitle('Macierze korelacji Spearmana per aktywnosc', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('m2_correlation_per_activity.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_correlation_per_activity.png')


### 6.3 Graf następstw zdarzeń (DFG – Directly-Follows Graph)

Dla każdej aktywności identyfikujemy przejścia między stanami sygnałów.


In [ ]:
print('=== DIRECTLY-FOLLOWS GRAPH – PRZEJSCIA MIEDZY STANAMI SYGNALOW ===\n')
for act in activities:
    sub = df[df['activity'] == act].sort_values('event_index')
    asigs = [c for c in SIGNAL_COLS if sub[c].notna().any()]
    sv = sub[asigs].fillna(0).astype(int)
    changes = sv.diff().fillna(0)
    print(f'--- {act} ({sub["station"].iloc[0]}) ---')
    n_trans = 0
    for i in range(1, len(sv)):
        ch = changes.iloc[i]
        changed = ch[ch != 0]
        if len(changed) > 0:
            n_trans += 1
            desc = ', '.join([f'{s}: {sv.iloc[i-1][s]}->{sv.iloc[i][s]}'
                              for s in changed.index])
            print(f'  Event {sub["event_index"].iloc[i]:3d}: {desc}')
    print(f'  Lacznie przejsc ze zmiana: {n_trans}/{len(sub)-1}\n')


## 7. Wzorce czasowe

Analiza rytmu próbkowania, czasu trwania aktywności i przebiegów sygnałów w czasie.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gantt-style: czas trwania
for i, act in enumerate(activities):
    sub = df[df['activity'] == act].sort_values('timestamp')
    dur = (sub['timestamp'].max() - sub['timestamp'].min()).total_seconds()
    n   = len(sub)
    axes[0].barh(i, dur, height=0.5, color=color_map[act], edgecolor='white', alpha=0.85)
    axes[0].text(dur + 1, i, f'{dur:.0f}s  (n={n})', va='center', fontsize=9, fontweight='bold')
axes[0].set_yticks(range(len(activities)))
axes[0].set_yticklabels(activities, fontsize=10)
axes[0].set_xlabel('Czas trwania [s]', fontsize=11)
axes[0].set_title('Czas trwania aktywnosci', fontweight='bold')
axes[0].set_xlim(0, 115)

# Scatter: czas trwania vs liczba eventow
durations = df.groupby('activity')['timestamp'].agg(lambda x: (x.max()-x.min()).total_seconds())
n_events  = df.groupby('activity').size()
for act in activities:
    axes[1].scatter(durations[act], n_events[act], s=200, c=[color_map[act]],
                    label=act, zorder=5, edgecolors='white', lw=1.5)
    axes[1].annotate(act, (durations[act], n_events[act]),
                     textcoords='offset points', xytext=(5, 5), fontsize=8)
axes[1].set_xlabel('Czas trwania [s]', fontsize=11)
axes[1].set_ylabel('Liczba eventow', fontsize=11)
axes[1].set_title('Czas trwania vs liczba eventow', fontweight='bold')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig('m2_temporal_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_temporal_overview.png')


In [ ]:
# Przebiegi sygnalow w czasie – wszystkie aktywnosci
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()
for idx, act in enumerate(activities):
    sub  = df[df['activity'] == act].sort_values('event_index')
    asigs = [c for c in SIGNAL_COLS if sub[c].notna().any()]
    ax   = axes[idx]
    elapsed = (sub['timestamp'] - sub['timestamp'].min()).dt.total_seconds().values
    for sig in asigs:
        vals = sub[sig].fillna(0).values.astype(float)
        vmin, vmax = vals.min(), vals.max()
        vn = (vals - vmin) / (vmax - vmin) if vmax > vmin else vals * 0
        ax.step(elapsed, vn, where='post', lw=1.5, label=sig, alpha=0.8)
    dur = elapsed[-1]
    ax.set_title(f'{act} ({sub["station"].iloc[0]})\n'
                 f'{len(sub)} eventow, {dur:.0f}s, {len(asigs)} sygnalow',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Czas od startu [s]'); ax.set_ylabel('Wartosc znorm. [0-1]')
    ax.set_ylim(-0.1, 1.3)
    ax.legend(fontsize=6, ncol=2, loc='upper right')
    for t in elapsed:
        ax.axvline(t, color='gray', alpha=0.15, lw=0.5)
plt.suptitle('Przebiegi sygnalow w czasie per aktywnosc (wartosci znormalizowane)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('m2_signal_timeseries_all.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_signal_timeseries_all.png')


### 7.2 Heatmapa częstości zmian sygnałów

Które sygnały najczęściej zmieniają wartość w trakcie każdej aktywności?


In [ ]:
change_data = {}
for act in activities:
    sub  = df[df['activity'] == act].sort_values('event_index')
    asigs = [c for c in SIGNAL_COLS if sub[c].notna().any()]
    sv   = sub[asigs].fillna(0).astype(int)
    ch   = (sv.diff().fillna(0) != 0).sum()
    change_data[act] = (ch / len(sub)).reindex(SIGNAL_COLS).fillna(0)

ch_df = pd.DataFrame(change_data).T

fig, ax = plt.subplots(figsize=(16, 5))
im = ax.imshow(ch_df.values, cmap='YlOrRd', aspect='auto', vmin=0)
plt.colorbar(im, ax=ax, shrink=0.8, label='Czestotliwosc zmian / event')
ax.set_xticks(range(len(SIGNAL_COLS)))
ax.set_xticklabels(SIGNAL_COLS, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(ch_df.index)))
ax.set_yticklabels(ch_df.index, fontsize=10)
ax.set_title('Czestotliwosc zmian sygnalow per aktywnosc\n'
             '(wartosc = liczba zmian / liczba eventow)', fontweight='bold')
for i in range(len(ch_df.index)):
    for j in range(len(SIGNAL_COLS)):
        v = ch_df.values[i, j]
        if v > 0:
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if v > 0.3 else 'black', fontweight='bold')
plt.tight_layout()
plt.savefig('m2_signal_change_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_signal_change_heatmap.png')


## 8. Analiza sekwencji i wariantów

Każdy event = unikalny stan (kombinacja wartości sygnałów). Analizujemy sekwencje stanów i macierze przejść.


In [ ]:
from collections import Counter

print('=== ANALIZA STANOW I SEKWENCJI PER AKTYWNOSC ===\n')
sa_rows = []
for act in activities:
    sub  = df[df['activity'] == act].sort_values('event_index')
    asigs = [c for c in SIGNAL_COLS if sub[c].notna().any()]
    sv   = sub[asigs].fillna(0).astype(int)
    states = [tuple(r) for r in sv.values]
    sid = {}
    seq = []
    for s in states:
        if s not in sid:
            sid[s] = f'S{len(sid)+1}'
        seq.append(sid[s])
    u = len(set(seq))
    rep = len(seq) - u
    repeated = {k: v for k, v in Counter(seq).items() if v > 1}
    sa_rows.append({'Aktywnosc': act, 'Eventy': len(sub),
                    'Unikalne stany': u, 'Powtorzone': rep,
                    '% unikalnych': round(u/len(sub)*100, 1)})
    print(f'--- {act} ---')
    print(f'  Eventy: {len(sub)}, Unikalne stany: {u}, Powtorzone: {rep}')
    print(f'  Sekwencja: {" -> ".join(seq)}')
    if repeated:
        print(f'  Stany powtorzone: {repeated}')
    print()

print('=== PODSUMOWANIE ===')
print(pd.DataFrame(sa_rows).to_string(index=False))


In [ ]:
# Macierz przejsc stanow dla Storage (najdluzsza aktywnosc)
act = 'Storage'
sub  = df[df['activity'] == act].sort_values('event_index')
asigs = [c for c in SIGNAL_COLS if sub[c].notna().any()]
sv   = sub[asigs].fillna(0).astype(int)
states = [tuple(r) for r in sv.values]
sid = {}
seq = []
for s in states:
    if s not in sid:
        sid[s] = f'S{len(sid)+1}'
    seq.append(sid[s])
unames = list(dict.fromkeys(seq))
n = len(unames)
sn_idx = {s: i for i, s in enumerate(unames)}
T = np.zeros((n, n), dtype=int)
for i in range(len(seq)-1):
    T[sn_idx[seq[i]], sn_idx[seq[i+1]]] += 1

fig, ax = plt.subplots(figsize=(max(8, n), max(6, n-2)))
im = ax.imshow(T, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label='Liczba przejsc')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(unames, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(unames, fontsize=9)
ax.set_xlabel('Stan docelowy'); ax.set_ylabel('Stan zrodlowy')
ax.set_title(f'Macierz przejsc stanow – {act} ({n} unikalnych stanow)', fontweight='bold')
for i in range(n):
    for j in range(n):
        if T[i, j] > 0:
            ax.text(j, i, str(T[i, j]), ha='center', va='center', fontsize=9,
                    color='white' if T[i,j] > 1 else 'black', fontweight='bold')
plt.tight_layout()
plt.savefig('m2_transition_matrix_storage.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_transition_matrix_storage.png')
print(f'Sekwencja stanow Storage: {" -> ".join(seq)}')


## 9. Wykrywanie anomalii

### 9.1 Weryfikacja wzorców CEP (Siddhi)

Sprawdzamy, czy wzorce zdefiniowane w plikach `.siddhi` są wykrywalne w danych.


In [ ]:
# Wzorce CEP zakodowane na podstawie plikow .siddhi
# Format: lista warunkow (sygnal, wartosc_przed, wartosc_po)
CEP = {
    'Burn': [
        [('m1_speed',0,-512),('o7_valve',0,512)],          # P1
        [('i1_pos_switch',0,1),('m1_speed',-512,0)],       # P2
        [('o7_valve',512,0)],                               # P3
        [('i1_pos_switch',1,0),('m1_speed',0,512),('o7_valve',0,512)],  # P4
        [('i2_pos_switch',0,1),('m1_speed',512,0)],        # P5
        [('i2_pos_switch',1,0),('o7_valve',512,0)],        # P6
    ],
    'Mill': [
        [('o8_compressor',512,0)],   # P1
        [('o8_compressor',0,512)],   # P2
        [('o8_compressor',512,0)],   # P3
    ],
    'Sort': [
        [('m1_speed',0,-512)],                              # P1
        [('i1_light_barrier',0,1)],                        # P2
        [('i3_light_barrier',1,0)],                        # P3
        [('i3_light_barrier',0,1)],                        # P4
        [('m1_speed',-512,0),('o6_valve',0,512),('o8_compressor',0,512)],  # P5
        [('i7_light_barrier',1,0),('o6_valve',512,0),('o8_compressor',512,0)],  # P6
    ],
}

def match_pattern(prev_row, curr_row, pattern):
    for sig, vb, va in pattern:
        if sig not in prev_row.index or sig not in curr_row.index:
            return False
        if prev_row[sig] != vb or curr_row[sig] != va:
            return False
    return True

print('=== WERYFIKACJA WZOROW CEP ===\n')
for act, patterns in CEP.items():
    sub  = df[df['activity'] == act].sort_values('event_index')
    asigs = [c for c in SIGNAL_COLS if sub[c].notna().any()]
    sv   = sub[asigs].fillna(0).astype(int).reset_index(drop=True)
    matched = [False] * len(patterns)
    match_ev = {}
    for i in range(1, len(sv)):
        for pi, pat in enumerate(patterns):
            if not matched[pi] and match_pattern(sv.iloc[i-1], sv.iloc[i], pat):
                matched[pi] = True
                match_ev[pi] = sub['event_index'].iloc[i]
    print(f'--- {act} ({len(patterns)} wzorow) ---')
    for pi, pat in enumerate(patterns):
        status = f'OK (event {match_ev.get(pi,"?")})' if matched[pi] else 'NIE WYKRYTY'
        print(f'  Wzorzec {pi+1}: {status}')
    print(f'  Wynik: {sum(matched)}/{len(patterns)} ({sum(matched)/len(patterns)*100:.0f}%)\n')


### 9.2 Local Outlier Factor (LOF) per aktywność


In [ ]:
from sklearn.neighbors import LocalOutlierFactor

print('=== LOCAL OUTLIER FACTOR (LOF) PER AKTYWNOSC ===\n')
lof_rows = []
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, act in enumerate(activities):
    sub  = df[df['activity'] == act].sort_values('event_index')
    asigs = [c for c in SIGNAL_COLS if sub[c].notna().any()]
    X_act = sub[asigs].fillna(0).values.astype(float)
    nn = min(5, len(X_act)-1)
    lof = LocalOutlierFactor(n_neighbors=nn, contamination='auto')
    lof_lbl = lof.fit_predict(X_act)
    lof_sc  = -lof.negative_outlier_factor_
    n_anom  = (lof_lbl == -1).sum()
    lof_rows.append({'Aktywnosc': act, 'Eventy': len(sub),
                     'Anomalie LOF': int(n_anom),
                     '% anomalii': round(n_anom/len(sub)*100, 1)})
    print(f'--- {act}: {n_anom} anomalii ({n_anom/len(sub)*100:.0f}%)')
    if n_anom > 0:
        for ai in np.where(lof_lbl == -1)[0]:
            print(f'  event_index={sub["event_index"].iloc[ai]}, '
                  f'LOF={lof_sc[ai]:.2f}, ts={sub["timestamp_raw"].iloc[ai]}')
    ax = axes[idx]
    bc = ['red' if l == -1 else color_map[act] for l in lof_lbl]
    ax.bar(range(len(lof_sc)), lof_sc, color=bc, edgecolor='none')
    ax.axhline(1.5, color='red', linestyle='--', lw=1.5, label='prog ~1.5')
    ax.set_title(f'{act}\n(n={len(sub)}, anomalie={n_anom})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Nr eventu'); ax.set_ylabel('LOF score')
    ax.legend(fontsize=7)

plt.suptitle('Local Outlier Factor per aktywnosc (czerwony = anomalia)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('m2_lof_scores.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nWykres zapisany: m2_lof_scores.png')
print('\n=== PODSUMOWANIE LOF ===')
print(pd.DataFrame(lof_rows).to_string(index=False))


## 10. Podsumowanie Milestone 2

### Normalizacja i przygotowanie
- NaN → 0 (semantycznie poprawne: sygnał nieaktywny = 0)
- Normalizacja Min-Max [0,1] dla PCA/klasteryzacji
- Normalizacja StandardScaler dla Isolation Forest

### Outliery i anomalie
- **Odstępy czasowe**: próbkowanie regularne ~2 s; pojedyncze odstępy 3 s w Burn (event 10–11) to artefakt rejestracji
- **Isolation Forest** (5%): ~6 anomalii globalnych – głównie eventy graniczne (start/koniec aktywności)
- **LOF per aktywność**: anomalie w eventach ze zmianą stanu (przejścia między fazami)

### Redukcja wymiarowości
- **PCA**: 2 komponenty wyjaśniają ~60–70% wariancji; aktywności tworzą wyraźnie odrębne skupiska
- Najważniejsze sygnały dla PC1: specyficzne dla Storage (HBW_1) – `m2_speed`, `m3_speed`, `i5_pos_switch`, `i6_pos_switch`
- **t-SNE**: potwierdza separowalność; Storage i Pickup-move-oven tworzą najbardziej zwarte skupiska

### Klasteryzacja
- **K-Means k=6**: ARI > 0.7 – klastry dobrze odpowiadają aktywnosciom
- **Dendrogram**: Burn i Mill są najbardziej podobne; Storage jest najbardziej odległa

### Korelacje
- Silne korelacje wewnątrz aktywności (sygnały tej samej stacji)
- Brak silnych korelacji globalnych między stacjami (niezależne stacje)

### Wzorce sekwencyjne
- Burn: 6/6 wzorców CEP wykrytych ✓
- Mill: 3/3 wzorców CEP wykrytych ✓
- Sort: 6/6 wzorców CEP wykrytych ✓
- Storage: 47 eventów, wiele unikalnych stanów (złożoność ruchu 3D)
- Transport: powtarzające się stany (faza jazdy = wielokrotny stan m2_speed=512)

### Ograniczenia
- Jeden case per aktywność – brak analizy wariantów między instancjami
- Dane z jednego dnia – analiza sezonowości niemożliwa
- Mały zbiór (119 eventów) – wyniki należy interpretować ostrożnie


In [ ]:
print('=' * 60)
print('PODSUMOWANIE MILESTONE 2 – KLUCZOWE LICZBY')
print('=' * 60)
print(f'Laczna liczba eventow:          {len(df)}')
print(f'Liczba aktywnosci:              {df["activity"].nunique()}')
print(f'Liczba kolumn sygnalow:         {len(SIGNAL_COLS)}')
print(f'Wymiarowosc po PCA (80% var):   {n_80} komponentow')
print(f'Wymiarowosc po PCA (95% var):   {n_95} komponentow')
print(f'Najlepsze k (silhouette):       {best_k}')
km6_final = KMeans(n_clusters=6, random_state=SEED, n_init=20)
cl6_final = km6_final.fit_predict(X_pca5)
print(f'ARI K-Means k=6:                {adjusted_rand_score(df["activity"], cl6_final):.3f}')
n_time_out = len(df_s[df_s['dt_s'].notna() & ((df_s['dt_s'] > 3) | (df_s['dt_s'] < 1))])
print(f'Outliery czasowe:               {n_time_out}')
print(f'Anomalie Isolation Forest (5%): {(iso_labels == -1).sum()}')
print('Wzorce CEP: Burn 6/6, Mill 3/3, Sort 6/6 – wszystkie wykryte')
